# GitHub Refactoring Dataset Scraper
Interactive notebook for scraping high-quality Python refactoring pairs from GitHub.

Focus: Larger, more meaningful refactors (>=10 lines changed) for small models (30M).

## 1. Imports

In [28]:
import os
import ast
from random import random
import sys
import json
import time
import logging
import difflib
import textwrap
import re
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Optional
import requests
from github import Github, GithubException, RateLimitExceededException, Auth
from tqdm import tqdm

print("✅ All imports successful")

✅ All imports successful


## 2. Configuration (TUNE THESE)

Adjust these parameters to control quality vs coverage.

In [29]:
# ===== QUALITY GATES (adjust for your model) =====
MIN_DIFF_LINES = 6       # Minimum lines changed (was 3, now 6 for better quality)
MAX_DIFF_LINES = 80        # Maximum lines changed
SMALL_DIFF_PROB = 0.3      # Probability to keep diffs <15 lines (0.0 = none, 1.0 = all)

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")  # or pass directly: "ghp_xxxx"
OUTPUT_FILE = "./dataset_quality.jsonl"    # Change output filename here
STATE_FILE = "./scrape_state.json"         # Progress checkpoint file
MAX_REPOS = 500                            # Total repos to target across all runs
MAX_REPOS_PER_RUN = 10                     # Commit progress every 10 processed repos
MAX_COMMITS = 500                        # Commits per repo
MAX_RUN_SECONDS = 21000                    # ~5h50m per run (leave headroom for workflow timeout)

# Commit message filters
KEEP_KEYWORDS = [
    "refactor", "refactoring", "clean up", "cleanup", "simplif",
    "rename", "extract method", "extract function", "deduplic",
    "restructur", "reorgani", "tidy", "readab",
]

REJECT_KEYWORDS = [
    "fix", "bug", "hotfix", "patch", "test", "feature", "feat",
    "add ", "new ", "implement", "merge", "revert", "wip",
    "release", "version", "bump", "update depend", "typo",
]

# Other filters
MAX_FUNCTIONS_CHANGED = 3
MIN_REPO_STARS = 50
MIN_REPO_AGE_DAYS = 365 * 5

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

print(f"Config loaded:")
print(f"  MIN_DIFF_LINES: {MIN_DIFF_LINES}")
print(f"  MAX_DIFF_LINES: {MAX_DIFF_LINES}")
print(f"  SMALL_DIFF_PROB: {SMALL_DIFF_PROB}")
print(f"  MIN_REPO_AGE_DAYS: {MIN_REPO_AGE_DAYS}")
print(f"  STATE_FILE: {STATE_FILE}")
print(f"  MAX_REPOS_PER_RUN: {MAX_REPOS_PER_RUN}")
print(f"  MAX_COMMITS: {MAX_COMMITS}")
print(f"  MAX_RUN_SECONDS: {MAX_RUN_SECONDS}")

Config loaded:
  MIN_DIFF_LINES: 6
  MAX_DIFF_LINES: 80
  SMALL_DIFF_PROB: 0.3
  MIN_REPO_AGE_DAYS: 1825
  STATE_FILE: ./scrape_state.json
  MAX_REPOS_PER_RUN: 10
  MAX_COMMITS: 500
  MAX_RUN_SECONDS: 21000


## 3. Data Model

In [30]:
@dataclass
class RefactorPair:
    before: str
    after: str
    refactor_type: str
    repo: str
    commit_sha: str
    file: str
    diff_lines: int

print("✅ RefactorPair dataclass defined")

✅ RefactorPair dataclass defined


## 4. Commit Message Filter

In [31]:
def message_is_refactor(message: str) -> bool:
    """Return True only if the commit message looks like a pure refactor."""
    msg = message.lower()

    if not any(kw in msg for kw in KEEP_KEYWORDS):
        return False

    if any(kw in msg for kw in REJECT_KEYWORDS):
        return False

    return True

print("✅ message_is_refactor defined")

✅ message_is_refactor defined


## 5. GitHub File Fetching

In [32]:
def fetch_file_at_commit(session: requests.Session, repo_full_name: str,
                          file_path: str, ref: str, token: str) -> Optional[str]:
    """Fetch raw file content from GitHub at a specific commit SHA."""
    url = f"https://raw.githubusercontent.com/{repo_full_name}/{ref}/{file_path}"
    headers = {"Authorization": f"token {token}"}
    try:
        resp = session.get(url, headers=headers, timeout=10)
        if resp.status_code == 200:
            return resp.text
        return None
    except requests.RequestException:
        return None

print("✅ fetch_file_at_commit defined")

✅ fetch_file_at_commit defined


## 6. AST Function Extraction

In [33]:
def extract_functions(source: str) -> dict[str, ast.FunctionDef]:
    """Parse Python source and return a dict of {function_name: node}."""
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return {}

    funcs = {}
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            funcs[node.name] = node
    return funcs

def node_to_source(source_lines: list[str], node: ast.FunctionDef) -> str:
    """Extract the exact source lines for a function node."""
    start = node.lineno - 1
    end = node.end_lineno
    return textwrap.dedent("".join(source_lines[start:end]))

print("✅ extract_functions and node_to_source defined")

✅ extract_functions and node_to_source defined


## 7. Diff and Filter Utilities

In [34]:
def count_changed_lines(before: str, after: str) -> int:
    """Count number of added + removed lines between two strings."""
    diff = list(difflib.unified_diff(
        before.splitlines(), after.splitlines(), lineterm=""
    ))
    changed = sum(
        1
        for line in diff
        if line.startswith(("+", "-"))
        and not line.startswith(("+++", "---"))
    )
    return changed

def diff_is_in_range(before: str, after: str) -> bool:
    n = count_changed_lines(before, after)
    return MIN_DIFF_LINES <= n <= MAX_DIFF_LINES

print("✅ count_changed_lines and diff_is_in_range defined")

✅ count_changed_lines and diff_is_in_range defined


## 8. Semantic Filters

In [35]:
def signatures_match(before_node: ast.FunctionDef,
                      after_node: ast.FunctionDef) -> bool:
    """Ensure the function signature didn't change."""
    ba = before_node.args
    aa = after_node.args

    before_argc = len(ba.args) + len(ba.posonlyargs) + len(ba.kwonlyargs)
    after_argc  = len(aa.args) + len(aa.posonlyargs) + len(aa.kwonlyargs)

    if before_node.name != after_node.name:
        return False
    if before_argc != after_argc:
        return False
    return True

def imports_unchanged(before_src: str, after_src: str) -> bool:
    """Reject pairs where new import statements were introduced."""
    def get_imports(src):
        try:
            tree = ast.parse(src)
        except SyntaxError:
            return set()
        imports = set()
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                for alias in node.names:
                    imports.add(alias.name)
            elif isinstance(node, ast.ImportFrom):
                imports.add(f"{node.module}.{[a.name for a in node.names]}")
        return imports

    before_imports = get_imports(before_src)
    after_imports  = get_imports(after_src)
    new_imports = after_imports - before_imports
    return len(new_imports) == 0

print("✅ signatures_match and imports_unchanged defined")

✅ signatures_match and imports_unchanged defined


## 9. Refactor Type Classification

In [36]:
def classify_refactor(before: str, after: str,
                       before_node: ast.FunctionDef,
                       after_node:  ast.FunctionDef) -> str:
    """Heuristic classifier for refactor type."""
    before_lines = before.splitlines()
    after_lines  = after.splitlines()

    def normalise_ids(src):
        return re.sub(r'\b[a-zA-Z_]\w*\b', 'X', src)

    if normalise_ids(before) == normalise_ids(after):
        return "rename"

    before_funcs = sum(1 for n in ast.walk(before_node)
                       if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)))
    after_funcs  = sum(1 for n in ast.walk(after_node)
                       if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)))
    if after_funcs > before_funcs:
        return "extract"

    before_nodes = sum(1 for _ in ast.walk(before_node))
    after_nodes  = sum(1 for _ in ast.walk(after_node))
    if after_nodes < before_nodes * 0.85:
        return "simplify"

    def count_duplicate_lines(lines):
        counts = {}
        for line in lines:
            stripped = line.strip()
            if len(stripped) > 10:
                counts[stripped] = counts.get(stripped, 0) + 1
        return sum(v - 1 for v in counts.values() if v > 1)

    before_dups = count_duplicate_lines(before_lines)
    after_dups  = count_duplicate_lines(after_lines)
    if before_dups > 0 and after_dups < before_dups:
        return "deduplicate"

    return "general"

print("✅ classify_refactor defined")

✅ classify_refactor defined


## 10. Main Commit Processor

In [37]:
def process_commit(commit, repo, session: requests.Session,
                   token: str) -> list[RefactorPair]:
    """Process a single commit and extract refactoring pairs."""
    pairs = []

    try:
        changed_files = [f for f in commit.files
                         if f.filename.endswith(".py")
                         and f.status in ("modified",)
                         and f.patch is not None]
    except Exception:
        return []

    if not changed_files:
        return []

    parent_sha = commit.parents[0].sha if commit.parents else None
    if parent_sha is None:
        return []

    for f in changed_files:
        before_src = fetch_file_at_commit(session, repo.full_name,
                                          f.filename, parent_sha, token)
        after_src  = fetch_file_at_commit(session, repo.full_name,
                                          f.filename, commit.sha, token)
        if not before_src or not after_src:
            continue

        before_funcs = extract_functions(before_src)
        after_funcs  = extract_functions(after_src)

        if not before_funcs or not after_funcs:
            continue

        common = set(before_funcs) & set(after_funcs)
        changed_count = 0

        before_lines = before_src.splitlines(keepends=True)
        after_lines  = after_src.splitlines(keepends=True)

        for func_name in common:
            bn = before_funcs[func_name]
            an = after_funcs[func_name]

            before_code = node_to_source(before_lines, bn)
            after_code  = node_to_source(after_lines, an)

            if before_code.strip() == after_code.strip():
                continue

            if not diff_is_in_range(before_code, after_code):
                continue

            if not signatures_match(bn, an):
                continue

            if not imports_unchanged(before_src, after_src):
                continue

            changed_count += 1
            if changed_count > MAX_FUNCTIONS_CHANGED:
                break

            rtype = classify_refactor(before_code, after_code, bn, an)
            diff_n = count_changed_lines(before_code, after_code)

            # Probabilistically filter small diffs for quality
            if diff_n < 15:
                if random() > SMALL_DIFF_PROB:
                    continue

            pairs.append(RefactorPair(
                before=before_code.strip(),
                after=after_code.strip(),
                refactor_type=rtype,
                repo=repo.full_name,
                commit_sha=commit.sha,
                file=f.filename,
                diff_lines=diff_n,
            ))

    return pairs

print("✅ process_commit defined")

✅ process_commit defined


## 11. Repo Search and Rate Limiting

In [38]:
def iter_repos(gh: Github, max_repos: int):
    """Search GitHub for high-quality Python repos."""
    queries = [
        "language:python stars:>200 pushed:>2022-01-01",
        "language:python stars:50..200 pushed:>2022-01-01",
        "language:python topic:web-framework stars:>100",
        "language:python topic:machine-learning stars:>100",
        "language:python topic:data-science stars:>50",
    ]

    seen = set()
    count = 0

    for query in queries:
        if count >= max_repos:
            break
        try:
            results = gh.search_repositories(query, sort="updated")
            for repo in results:
                if count >= max_repos:
                    break
                if repo.full_name in seen:
                    continue
                seen.add(repo.full_name)

                age_days = (time.time() - repo.created_at.timestamp()) / 86400
                if repo.stargazers_count < MIN_REPO_STARS:
                    continue
                if age_days < MIN_REPO_AGE_DAYS:
                    continue

                count += 1
                yield repo

        except RateLimitExceededException:
            wait_for_rate_limit(gh)
        except GithubException as e:
            log.warning(f"Repo search error: {e}")

def wait_for_rate_limit(gh: Github):
    """Block until GitHub rate limit resets."""
    reset_time = gh.get_rate_limit().core.reset.timestamp()
    wait = max(0, reset_time - time.time()) + 5
    log.warning(f"Rate limit hit — sleeping {wait:.0f}s")
    time.sleep(wait)

print("✅ iter_repos and wait_for_rate_limit defined")

✅ iter_repos and wait_for_rate_limit defined


## 12. RUN THE SCRAPER

Set your GitHub token and output path, then execute this cell.

In [39]:
# Configuration for this run

if not GITHUB_TOKEN:
    print("❌ ERROR: Set GITHUB_TOKEN environment variable")
    print("   On Windows PowerShell: $env:GITHUB_TOKEN='ghp_....'")
    print("   Or pass it directly: GITHUB_TOKEN = 'ghp_....'")
else:
    print(f"✅ Token configured")
    print(f"Output: {OUTPUT_FILE}")
    print(f"Scanning: {MAX_REPOS} repos × {MAX_COMMITS} commits")

✅ Token configured
Output: ./dataset_quality.jsonl
Scanning: 500 repos × 500 commits


In [ ]:
# Main scrape loop with checkpointed resume (safe for retries / scheduled runs)

def load_state(state_path: Path) -> dict:
    if not state_path.exists() or state_path.stat().st_size == 0:
        return {
            "processed_repos": [],
            "written": 0,
            "skipped": 0,
            "message_matched": 0,
            "candidate_commits": 0,
            "commit_errors": 0,
            "type_counts": {},
        }
    try:
        with state_path.open("r", encoding="utf-8") as f:
            state = json.load(f)
        if not isinstance(state.get("processed_repos", []), list):
            state["processed_repos"] = []
        if not isinstance(state.get("type_counts", {}), dict):
            state["type_counts"] = {}
        return state
    except Exception as e:
        log.warning(f"Could not load state file {state_path}: {e}")
        return {
            "processed_repos": [],
            "written": 0,
            "skipped": 0,
            "message_matched": 0,
            "candidate_commits": 0,
            "commit_errors": 0,
            "type_counts": {},
        }

def save_state(state_path: Path, state: dict):
    state["updated_at"] = int(time.time())
    tmp = state_path.with_suffix(state_path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)
    tmp.replace(state_path)

gh = Github(auth=Auth.Token(GITHUB_TOKEN), per_page=100)
session = requests.Session()
session.headers["Accept"] = "application/vnd.github.v3.raw"

out_path = Path(OUTPUT_FILE)
state_path = Path(STATE_FILE)
state = load_state(state_path)
processed_repos = set(state.get("processed_repos", []))

written = int(state.get("written", 0))
skipped = int(state.get("skipped", 0))
message_matched = int(state.get("message_matched", 0))
candidate_commits = int(state.get("candidate_commits", 0))
commit_errors = int(state.get("commit_errors", 0))
type_counts = dict(state.get("type_counts", {}))

if written == 0 and out_path.exists() and out_path.stat().st_size > 0:
    # Fallback if output exists but state was reset/missing.
    with out_path.open("r", encoding="utf-8") as fin:
        written = sum(1 for _ in fin if _.strip())

run_started = time.time()
repos_processed_this_run = 0
max_repos_this_run = min(MAX_REPOS_PER_RUN, max(0, MAX_REPOS - len(processed_repos)))

log.info(f"Starting scrape chunk -> {out_path}")
log.info(f"Resume state from {state_path}; already processed repos: {len(processed_repos)}")
log.info(f"Chunk target repos this run: {max_repos_this_run}")
log.info(f"Quality filter: {MIN_DIFF_LINES} <= diff_lines <= {MAX_DIFF_LINES}")

if max_repos_this_run <= 0:
    log.info("Target reached already (MAX_REPOS). Nothing to do.")
else:
    with open(out_path, "a", encoding="utf-8") as fout:
        repo_iter = iter_repos(gh, MAX_REPOS * 2)

        for repo in tqdm(repo_iter, desc="repos", unit="repo"):
            if repo.full_name in processed_repos:
                continue

            if repos_processed_this_run >= max_repos_this_run:
                break

            if MAX_RUN_SECONDS is not None and (time.time() - run_started) >= MAX_RUN_SECONDS:
                log.info("Max runtime reached for this chunk; saving state and exiting.")
                break

            log.info(f"  -> {repo.full_name} (stars={repo.stargazers_count})")

            try:
                commits = repo.get_commits()
            except GithubException as e:
                log.warning(f"Could not read commits for {repo.full_name}: {e}")
                processed_repos.add(repo.full_name)
                repos_processed_this_run += 1
                state.update({
                    "processed_repos": sorted(processed_repos),
                    "written": written,
                    "skipped": skipped,
                    "message_matched": message_matched,
                    "candidate_commits": candidate_commits,
                    "commit_errors": commit_errors,
                    "type_counts": type_counts,
                })
                save_state(state_path, state)
                continue

            commit_count = 0
            for commit in commits:
                if commit_count >= MAX_COMMITS:
                    break
                commit_count += 1

                try:
                    remaining = gh.get_rate_limit().core.remaining
                    if remaining < 50:
                        wait_for_rate_limit(gh)
                except Exception:
                    pass

                try:
                    msg = commit.commit.message
                except Exception:
                    continue

                if not message_is_refactor(msg):
                    skipped += 1
                    continue

                message_matched += 1

                try:
                    pairs = process_commit(commit, repo, session, GITHUB_TOKEN)
                    candidate_commits += 1
                except RateLimitExceededException:
                    wait_for_rate_limit(gh)
                    continue
                except Exception as e:
                    commit_errors += 1
                    log.warning(f"Commit error {commit.sha[:7]} in {repo.full_name}: {e}")
                    continue

                for pair in pairs:
                    fout.write(json.dumps(asdict(pair), ensure_ascii=False) + "\n")
                    written += 1
                    type_counts[pair.refactor_type] = type_counts.get(pair.refactor_type, 0) + 1

                if pairs:
                    fout.flush()
                    log.info(f"    +{len(pairs)} pair(s) from {commit.sha[:7]} | total={written}")

            processed_repos.add(repo.full_name)
            repos_processed_this_run += 1

            state.update({
                "processed_repos": sorted(processed_repos),
                "written": written,
                "skipped": skipped,
                "message_matched": message_matched,
                "candidate_commits": candidate_commits,
                "commit_errors": commit_errors,
                "type_counts": type_counts,
            })
            save_state(state_path, state)

log.info("=" * 50)
log.info(f"Done chunk. {written} pairs written to {out_path}")
log.info(f"State saved to {state_path}")
log.info(f"Repos processed this run: {repos_processed_this_run}")
log.info(f"Total repos marked processed: {len(processed_repos)} / {MAX_REPOS}")
log.info(f"Skipped by message filter: {skipped} commits")
log.info(f"Commits passing message filter: {message_matched}")
log.info(f"Commits processed for extraction: {candidate_commits}")
log.info(f"Commit-level errors: {commit_errors}")
log.info("Breakdown by refactor type:")
for k, v in sorted(type_counts.items(), key=lambda x: -x[1]):
    log.info(f"  {k:15s}: {v}")

22:26:59  INFO  Starting scrape chunk -> dataset_quality.jsonl
22:26:59  INFO  Resume state from scrape_state.json; already processed repos: 0
22:26:59  INFO  Chunk target repos this run: 10
22:26:59  INFO  Quality filter: 6 <= diff_lines <= 80
repos: 0repo [00:00, ?repo/s]22:27:04  INFO    -> pypa/virtualenv (stars=5030)
22:28:22  INFO      +5 pair(s) from 4fb0401 | total=5
repos: 1repo [05:01, 301.77s/repo]22:32:01  INFO    -> marcizhu/marcizhu (stars=231)
repos: 2repo [09:19, 275.78s/repo]22:36:18  INFO    -> coleifer/scout (stars=315)
22:36:45  INFO      +1 pair(s) from 4571b9c | total=6
22:36:59  INFO      +1 pair(s) from 5ed9095 | total=7
<unknown>:3: SyntaxWarning: invalid escape sequence '\ '
<unknown>:3: SyntaxWarning: invalid escape sequence '\ '
repos: 3repo [11:54, 220.87s/repo]22:38:54  INFO    -> sunlabuiuc/PyHealth (stars=1517)
22:39:48  INFO      +1 pair(s) from 82011b0 | total=8
22:40:27  INFO      +1 pair(s) from 7937d77 | total=9
repos: 4repo [16:27, 241.34s/repo]22:

## 13. Validate Your Results

Run this to check what you collected.

In [ ]:
import pandas as pd
from pathlib import Path

path = Path(OUTPUT_FILE)
if not path.exists() or path.stat().st_size == 0:
    print(f"No data found in {path}. Run Cell 12 first.")
else:
    df = pd.read_json(path, lines=True)
    required_cols = {"diff_lines", "refactor_type", "repo"}
    missing = required_cols - set(df.columns)

    if missing:
        print(f"Unexpected schema. Missing columns: {sorted(missing)}")
        print(f"Columns found: {list(df.columns)}")
    else:
        print(f"\nDataset Statistics")
        print(f"Total pairs: {len(df)}")
        print(f"Diff lines range: {df['diff_lines'].min()} - {df['diff_lines'].max()}")
        print(f"Diff lines mean: {df['diff_lines'].mean():.1f}")
        print("\nRefactor type distribution:")
        print(df['refactor_type'].value_counts())
        print("\nTop 10 repos:")
        print(df['repo'].value_counts().head(10))


Dataset Statistics
Total pairs: 6
Diff lines range: 16 - 54
Diff lines mean: 27.3

Refactor type distribution:
refactor_type
deduplicate    3
general        3
Name: count, dtype: int64

Top 10 repos:
repo
rtCamp/Frappe-Manager    6
Name: count, dtype: int64
